Some preliminaries to load the libraries for running experiments

In [2]:
# autoload
%load_ext autoreload
%autoreload 2

# import necessary libraries
import numpy as np

# Load PGL libraries and start a PGL window
from pgl import pgl, pglTask, pglExperiment, pglParameter, pglParameterBlock, pglTestTask
pgl = pgl()

# close any existing windows
pgl.cleanUp()

================================ pglBase: init =================================
(pgl) mglMetal error log can be viewed in MacOS Console app by searching for PROCESS mglMetal or in a terminal with:
      log stream --level info --process mglMetal
(pgl) To search for something specifc, e.g. messages from mglMovie:
      log stream --predicate 'eventMessage CONTAINS "mglMovie"' --style syslog --level info
(pgl:checkOS) Python version: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:35:20) [Clang 16.0.6 ]
(pgl:checkOS) Running on MacBook Pro (MacBookPro18,3) with macOS version: 26.3.1
(pgl:checkOS) Apple M1 Pro Cores: 8 (6 Performance and 2 Efficiency) Memory: 32 GB
(pgl:checkOS) GPU: Apple M1 Pro (Built-In) 14 cores, Metal 4 support
(pgl:checkOS)   Color LCD [Main Display]: 3024 x 1964 Retina (Built-in Liquid Retina XDR Display) GammaTable size: 1024
(pgl:checkOS)   LED Cinema Display: 1920 x 1200 (WUXGA - Widescreen Ultra eXtended Graphics Array) (Unknown type) GammaTable size

Here we define the task whcih is run by the experiment. It defines how each trial is structured (into segments like fixation, stimulus display and response), what the parameters of the experiment are (e.g. motion coherences to test) what to draw on the screen, and how to handle subject responses.

In [3]:
class pglDetectionTask(pglTask):
    
    ########################
    def __init__(self, pgl):
        super().__init__(pgl)
        
        # set task name - this is the readable format. 
        # note that everything in self.settings, self.state and self.data
        # get automatically stored in json files after the experiment is finished
        # and can be read back by doing
        # e = pglExperiment(experimentName="nameOfExperiment") where nameOfExperiment
        # is set below when you instantiate a pglExperiment
        self.settings.taskName = "Random Dot Motion Task"
        
        # segment lengths in seconds. THis is how long each segment of a trial
        # should run for. Here we have 3 segments of a trial: fixation, stimulus, response
        self.settings.seglen = [0.5, 0.2, 1.5]
        
        # fixed parameters, these are settings for the task and do not change
        # from trial-to-trial. Note that coherence and direction of the stimulus
        # will change from trial to trial, but is a setting here, as the next step of the code is
        # to assign a parameter to these ranges which will automatically
        # block randomize them from trial-to-trial
        self.settings.fixedParameters = {
            'width':15,
            'height':10,
            'coherence':(0.1,1),
            'dir':np.arange(0,360,45)
        }        
        p = self.settings.fixedParameters

        # These are parameters which will vary from trial-to-trial. 
        coherence = pglParameter('coherence',p['coherence'])
        dir = pglParameter('dir',p['dir'])
        self.addParameter(pglParameterBlock([dir, coherence]))
        
        # initalize stimulus, note that width and height are in degrees of visual angle
        self.rdk = pgl.randomDots(width=p['width'], height=p['height'])
    ########################
    def updateScreen(self):
        # what to display on the segment depends on what segment of the trial we are in
        if self.state.currentSegment==1:
            # for the stimulus segment, display the random dot kinematogram with the current parameters
            self.rdk.display(direction=self.currentParams['dir'], coherence=self.currentParams['coherence'], speed=5.0)

# initialize task
detectionTask = pglDetectionTask(pgl)


Here we create an experiment, attach the task defined above, and run the experiment. Note that it will automatically save the data at the end of the run.

In [ ]:
# Set up experiment
e = pglExperiment(pgl, "window", experimentName="detectionTask")
e.initScreen(backgroundColor=0.5)

# add the task
e.addTask(detectionTask)

# and run the experiment
e.run()

(pglSettingsManager:loadSettings) Loading settings from '/Users/justin/.pgl/settings/window.json'.
================================= pglBase:open =================================
(pglBase:open) Starting mglMetal application: /Users/justin/Library/Developer/Xcode/DerivedData/mglMetal-aklbakxihbwuyvekmqioljijytmv/Index.noindex/Build/Products/Debug/mglMetalUITests-Runner.app
(pglBase:open) Using socket with address: /Users/justin/Library/Containers/gru.mglMetal/Data/pglMetal.socket.20260325_100324.g8zheqJDtd
(pglBase:open) ❌ Error starting mglMetal application: Command '['open', '-g', '-n', '/Users/justin/Library/Developer/Xcode/DerivedData/mglMetal-aklbakxihbwuyvekmqioljijytmv/Index.noindex/Build/Products/Debug/mglMetalUITests-Runner.app', '--args', '-mglConnectionAddress', '/Users/justin/Library/Containers/gru.mglMetal/Data/pglMetal.socket.20260325_100324.g8zheqJDtd']' returned non-zero exit status 1.


The application cannot be opened because its executable is missing.
